# Modelling panel — feature preparation and handoff

Panel cleaning, locked unit-grain late priority species pivot, and the modelling feature matrix handoff for [`../04_modeling/`](../04_modeling/) and [`train.py`](../../train.py).

| Item | Detail |
|------|--------|
| Input | `modelling_panel.parquet` + `modelling_panel_species.parquet` + `modelling_panel_manifest.json` |
| Grain | Observed VectorNet × pathogen: Tier 1 `(h3_cell, year, month, vector_group, pathogen_name)`; Tier 2 adds `country_iso2` |
| Primary target | `pathogen_within_5y` (forward ≤5y, same key × **this** pathogen) |
| Features (vector) | Group aggregates + `species_richness`; priority `vector_count_<slug>` via **unit-grain pivot** then merge |
| Training filter (upstream) | Vector coverage required — virus-only units already excluded; no synthetic empty months |
| Spec | [`specs/03`](../../specs/03-spatial-temporal-matching-strategy.md) + [`specs/04`](../../specs/04-reference-data-and-panel-pipeline.md) |

**Prerequisite:** run [`../01_sources/10_merge_modelling_panel.ipynb`](../01_sources/10_merge_modelling_panel.ipynb) so the panel **and** species side table exist.

**Species features:** after panel filters, take distinct unit keys → restrict species-long → pivot closed priority taxa at **unit** grain → left-merge onto pathogen-long rows (never join species-long before pivot).

Descriptive EDA on the raw panel lives under [`../02_eda/`](../02_eda/). This notebook covers load → geo/time filters → unit-grain species pivot → modelling handoff (`X` / `y` / `meta`).


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

from vector_guard.data import default_processed_dir, find_repo_root
from vector_guard.panel_features import (
    SPECIES_LONG_COLS,
    UNIT_KEY_COLS,
    attach_priority_species_features,
    priority_count_columns,
    validate_species_long,
)
from vector_guard.resolve import H3_RESOLUTION, load_priority_vector_taxa

REPO_ROOT = find_repo_root(Path.cwd())
PROCESSED_DIR = default_processed_dir(REPO_ROOT)
PANEL_PATH = PROCESSED_DIR / "modelling_panel.parquet"
SPECIES_PATH = PROCESSED_DIR / "modelling_panel_species.parquet"
MANIFEST_PATH = PROCESSED_DIR / "modelling_panel_manifest.json"

# Modelling window: denser VectorNet era (inclusive).
YEAR_MIN = 2010
YEAR_MAX = None  # inclusive upper bound; None = no cap

# Winsorize heavy-tailed vector *effort* proxies only (not priority species counts).
COUNT_WINSOR_Q = 0.99
COUNT_COLS = [
    "vector_detection_count",
    "individual_count_sum",
    "individual_count_mean",
]

IDENTITY_COLS = [
    "spatial_tier",
    "h3_cell",
    "country_iso2",
    "year",
    "month",
    "vector_group",
    "pathogen_name",
]
TIER1_KEY = ["h3_cell", "year", "month", "vector_group", "pathogen_name"]
TIER2_KEY = ["country_iso2", "year", "month", "vector_group", "pathogen_name"]

# Primary supervised target (spec 04).
TARGET_COL = "pathogen_within_5y"
LABEL_COLS = [
    "pathogen_within_5y",
    "pathogen_within_10y",
    "pathogen_ever_before",
    "pathogen_ever_after",
    "pathogen_same_month",
    "pathogen_detection_count_within_5y",
    "pathogen_detection_count_within_10y",
    "years_to_first_pathogen_after",
    "years_since_last_pathogen_before",
]

# Locked priority taxa reference (closed set). Wide vector_count_* names and
# FEATURE_COLS are built *after* unit-grain pivot onto panel_prep — not assumed
# present on the raw merge panel.
_priority = load_priority_vector_taxa()
PRIORITY_COUNT_COLS: list[str] = []  # filled after attach_priority_species_features
FEATURE_COLS_BASE = [
    "vector_detection_count",
    "individual_count_sum",
    "individual_count_mean",
    "species_richness",
    "pathogen_ever_before",  # history before row time; not the forward target
]
FEATURE_COLS: list[str] = list(FEATURE_COLS_BASE)

# Forward-window QA columns (not modelling features — would leak label horizon).
SHARE_COLS = [
    "share_pathogen_exact_coords",
    "share_pathogen_subnational_centroid",
]

{
    "REPO_ROOT": str(REPO_ROOT),
    "PANEL_PATH": str(PANEL_PATH.relative_to(REPO_ROOT)),
    "SPECIES_PATH": str(SPECIES_PATH.relative_to(REPO_ROOT)),
    "H3_RESOLUTION": H3_RESOLUTION,
    "YEAR_MIN": YEAR_MIN,
    "n_priority_taxa_ref": len(_priority),
    "winsorize_cols": COUNT_COLS,
}


## Load panel, species side table, and merge manifest

Do **not** require `vector_count_*` on the raw panel. Priority species features are attached after filters via unit-grain pivot from `modelling_panel_species.parquet`.


In [ ]:
if not PANEL_PATH.exists():
    raise FileNotFoundError(
        f"Missing {PANEL_PATH}. Run notebooks/01_sources/10_merge_modelling_panel.ipynb first."
    )
if not SPECIES_PATH.exists():
    raise FileNotFoundError(
        f"Missing {SPECIES_PATH}. Run notebooks/01_sources/10_merge_modelling_panel.ipynb first."
    )

panel = pd.read_parquet(PANEL_PATH)
species_long = validate_species_long(pd.read_parquet(SPECIES_PATH))
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8")) if MANIFEST_PATH.exists() else {}

assert "pathogen_name" in panel.columns
assert TARGET_COL in panel.columns
assert "virus_within_5y" not in panel.columns
assert "pathogen_name" not in species_long.columns
assert list(species_long.columns) == SPECIES_LONG_COLS
# Do not assert PRIORITY_COUNT_COLS on the raw panel — they are built after pivot.

{
    "rows": len(panel),
    "cols": len(panel.columns),
    "species_rows": len(species_long),
    "species_units": int(species_long.loc[:, UNIT_KEY_COLS].drop_duplicates().shape[0]),
    "manifest_panel_rows": manifest.get("panel_rows"),
    "manifest_species_table_rows": manifest.get("species_table_rows"),
    "manifest_date_range": manifest.get("date_range"),
    "primary_label": manifest.get("primary_label"),
    "secondary_label": manifest.get("secondary_label"),
    "eligibility": manifest.get("eligibility"),
    "n_pathogens": manifest.get("n_pathogens"),
    "pathogen_positive_units": manifest.get("pathogen_positive_units"),
    "pathogen_positive_rate": manifest.get("pathogen_positive_rate"),
    "pathogen_within_10y_units": manifest.get("pathogen_within_10y_units"),
    "pathogen_ever_after_units": manifest.get("pathogen_ever_after_units"),
    "pathogen_ever_before_units": manifest.get("pathogen_ever_before_units"),
    "pathogen_same_month_units": manifest.get("pathogen_same_month_units"),
    "rows_by_spatial_tier": manifest.get("rows_by_spatial_tier"),
    "training_filter": manifest.get("training_filter"),
}


## Geo / time filters and modelling prep

Applied rules:

1. **Identity completeness** — drop rows missing required key fields for their `spatial_tier` (including `pathogen_name`).
2. **Valid calendar month** — `month` in 1–12.
3. **VectorNet coverage** — keep `vector_present == 1` (already true for the merge output).
4. **Modelling year window** — `year >= YEAR_MIN` (and `<= YEAR_MAX` if set).
5. **Feature prep** — winsorize vector effort `COUNT_COLS` at p99 only (not species `vector_count_*`). Forward-window `share_pathogen_*` stay nullable QA (not in `FEATURE_COLS`). Delay columns stay nullable and are not modelling features here.
6. **Species features (locked unit-grain pivot)** — use loaded `species_long` → distinct unit keys from filtered `panel_prep` → pivot **only** `priority_vector_taxa.csv` to `vector_count_<slug>` (0-fill) → left-merge onto pathogen-long rows → then set `PRIORITY_COUNT_COLS` / `FEATURE_COLS`. Row count must not change.

Row drops are reported; primary-label (`pathogen_within_5y`) retention is checked. Stratify downstream EDA / metrics by `pathogen_name`.


In [ ]:
def apply_panel_prep(
    df: pd.DataFrame,
    *,
    year_min: int = YEAR_MIN,
    year_max: int | None = YEAR_MAX,
    winsor_q: float = COUNT_WINSOR_Q,
) -> tuple[pd.DataFrame, dict]:
    """Filter and prepare the modelling panel; return (prepared_df, filter_report)."""
    work = df.copy()
    n0 = len(work)
    n_pos0 = int(work[TARGET_COL].sum())
    report: dict = {"n_input": n0, "n_pos_input": n_pos0}

    # 1. Identity by tier
    h3_ok = (work["spatial_tier"] == "h3") & work["h3_cell"].notna()
    country_ok = (work["spatial_tier"] == "country") & work["country_iso2"].notna()
    common_ok = (
        work["spatial_tier"].isin(["h3", "country"])
        & work["year"].notna()
        & work["month"].notna()
        & work["vector_group"].fillna("").ne("")
        & work["pathogen_name"].fillna("").ne("")
        & work["country_iso2"].notna()  # metadata preferred on Tier 1 too
    )
    mask_id = common_ok & (h3_ok | country_ok)
    report["dropped_identity"] = int((~mask_id).sum())
    work = work.loc[mask_id].copy()

    # 2. Valid month
    mask_month = work["month"].between(1, 12)
    report["dropped_bad_month"] = int((~mask_month).sum())
    work = work.loc[mask_month].copy()

    # 3. VectorNet coverage
    mask_vec = work["vector_present"] == 1
    report["dropped_no_vector"] = int((~mask_vec).sum())
    work = work.loc[mask_vec].copy()

    # 4. Year window
    mask_year = work["year"] >= year_min
    if year_max is not None:
        mask_year &= work["year"] <= year_max
    report["dropped_year_window"] = int((~mask_year).sum())
    report["year_min"] = year_min
    report["year_max"] = year_max
    work = work.loc[mask_year].copy()

    # 5. Feature prep — winsorize group effort COUNT_COLS only (not species counts).
    caps = {c: float(work[c].quantile(winsor_q)) for c in COUNT_COLS}
    report["winsor_caps"] = caps
    report["winsor_q"] = winsor_q
    report["winsorize_cols"] = list(COUNT_COLS)
    for col, cap in caps.items():
        work[f"{col}_raw"] = work[col]
        work[col] = work[col].clip(upper=cap)

    work["year"] = work["year"].astype(int)
    work["month"] = work["month"].astype(int)

    report["n_output"] = len(work)
    report["n_pos_output"] = int(work[TARGET_COL].sum())
    report["n_dropped_total"] = n0 - len(work)
    report["pos_retained"] = report["n_pos_output"] == n_pos0 or report[
        "n_pos_output"
    ] == int(df.loc[df["year"] >= year_min, TARGET_COL].sum())
    return work.reset_index(drop=True), report


panel_prep, filter_report = apply_panel_prep(panel)

# 6. Species: locked closed priority set via unit-grain pivot, then merge
# (species_long already loaded; never join long×long onto pathogen-long).
n_prep_before_species = len(panel_prep)
panel_prep = attach_priority_species_features(panel_prep, species_long, _priority)
assert len(panel_prep) == n_prep_before_species

# Build modelling feature list only after pivot+merge (not from raw panel columns).
PRIORITY_COUNT_COLS = priority_count_columns(_priority)
FEATURE_COLS = [
    "vector_detection_count",
    "individual_count_sum",
    "individual_count_mean",
    "species_richness",
    *PRIORITY_COUNT_COLS,
    "pathogen_ever_before",
]
missing_priority = [c for c in PRIORITY_COUNT_COLS if c not in panel_prep.columns]
assert not missing_priority, missing_priority
assert PRIORITY_COUNT_COLS == [
    c for c in panel_prep.columns if c.startswith("vector_count_")
], "modelling vector_count_* must match locked priority set exactly"

filter_report["species_source"] = "modelling_panel_species.parquet"
filter_report["species_long_rows"] = int(len(species_long))
filter_report["n_priority_features"] = len(PRIORITY_COUNT_COLS)
filter_report["n_units_prep"] = int(
    panel_prep.loc[:, UNIT_KEY_COLS].drop_duplicates().shape[0]
)
filter_report


In [ ]:
prep_check_cols = IDENTITY_COLS + FEATURE_COLS + [
    TARGET_COL,
    "pathogen_within_10y",
    "pathogen_detection_count_within_5y",
    "pathogen_detection_count_within_10y",
    "pathogen_same_month",
]
prep_missing = panel_prep[prep_check_cols].isna().sum()
assert int(prep_missing.sum()) == 0, prep_missing[prep_missing > 0].to_dict()
assert "virus_within_5y" not in panel_prep.columns
assert "virus_detected" not in panel_prep.columns

{
    "prep_rows": len(panel_prep),
    "prep_pos_5y": int(panel_prep[TARGET_COL].sum()),
    "prep_pos_rate_5y": float(panel_prep[TARGET_COL].mean()),
    "prep_within_10y": int(panel_prep["pathogen_within_10y"].sum()),
    "prep_ever_after": int(panel_prep["pathogen_ever_after"].sum()),
    "prep_n_pathogens": int(panel_prep["pathogen_name"].nunique()),
    "prep_year_range": [int(panel_prep["year"].min()), int(panel_prep["year"].max())],
    "prep_missing_feature_cells": int(prep_missing.sum()),
    "spatial_tier_counts": panel_prep["spatial_tier"].value_counts().to_dict(),
}


## Modelling handoff

`panel_prep` is the cleaned analysis table for [`../04_modeling/`](../04_modeling/) and [`train.py`](../../train.py).

| Object | Role |
|--------|------|
| `panel` | Raw merge output (pathogen-long; no merge-time `vector_count_*`) |
| `species_long` | Long composition from `modelling_panel_species.parquet` |
| `panel_prep` | Filtered + effort-winsorized + unit-pivoted priority species counts |
| `TARGET_COL` | `pathogen_within_5y` |
| `LABEL_COLS` | Forward / ever / delay / same-month label family (pathogen-specific) |
| `FEATURE_COLS` | Built **after** species pivot: effort + `species_richness` + locked `PRIORITY_COUNT_COLS` + `pathogen_ever_before` |
| `COUNT_COLS` | Only columns winsorized (group effort) — not `vector_count_*` |
| `filter_report` | Drop counts, winsor caps, species pivot metadata |

**Species side-table grain:** observed VectorNet **unit** × distinct `scientific_name` (**before** pathogen explosion). Schema is exactly `SPECIES_LONG_COLS` — **no** `pathogen_name`.

**Species pivot rule (locked for `train.py`):**
1. Filter panel (`apply_panel_prep`) — winsorize `COUNT_COLS` only.
2. Distinct `UNIT_KEY_COLS` from `panel_prep`.
3. Restrict `species_long` to those units.
4. Pivot closed `priority_vector_taxa.csv` → `vector_count_<feature_slug>` (absent → 0).
5. Left-merge wide counts onto pathogen-long rows (broadcast; row count unchanged).
6. Set `PRIORITY_COUNT_COLS` / `FEATURE_COLS` from that closed set.

Do not join species-long onto pathogen-long before pivoting. Do not winsorize priority species columns. Do not swap in frequency-selected taxa for the modelling handoff.

Use `pathogen_within_10y` as an alternate target for sensitivity runs. Stratify metrics by `pathogen_name`. Delay columns are for EDA histograms (`.dropna()`), not the primary supervised target.

Next: multi-model training under [`../04_modeling/`](../04_modeling/), then export via [`train.py`](../../train.py).


In [ ]:
X = panel_prep[FEATURE_COLS].copy()
y = panel_prep[TARGET_COL].astype(int).copy()
meta = panel_prep[
    IDENTITY_COLS
    + [
        "vector_present",
        "first_vector_year",
        "first_vector_month",
        "pathogen_detection_count_within_5y",
        "pathogen_detection_count_within_10y",
        "pathogen_within_10y",
        "pathogen_ever_before",
        "pathogen_ever_after",
        "pathogen_same_month",
        "years_to_first_pathogen_after",
        "years_since_last_pathogen_before",
    ]
].copy()

handoff = {
    "X_shape": X.shape,
    "y_shape": y.shape,
    "y_positive": int(y.sum()),
    "y_positive_rate": float(y.mean()),
    "n_pathogens": int(meta["pathogen_name"].nunique()),
    "feature_cols": FEATURE_COLS,
    "priority_count_cols": PRIORITY_COUNT_COLS,
    "winsorize_cols": COUNT_COLS,
    "target_col": TARGET_COL,
    "label_cols": LABEL_COLS,
    "species_path": str(SPECIES_PATH.relative_to(REPO_ROOT)),
    "filter_report": filter_report,
}
display(X[COUNT_COLS + ["species_richness", "pathogen_ever_before"]].describe().T)
handoff
